# Practice: Building a Model Validation & Drift Detection Test Suite

**Context.** You're a QA engineer embedded in a team that ships a breast-cancer
diagnosis classifier (malignant vs. benign) built on the classic Wisconsin
Diagnostic Breast Cancer dataset (`sklearn.datasets.load_breast_cancer`, a real
clinical dataset, 569 patients, 30 measured features per tumor). The team has
had two incidents in the past: (1) a feature pipeline silently started sending
out-of-range values after an upstream change, and (2) a model update that
looked fine on overall accuracy but quietly lost recall on the malignant class
— the costly kind of false negative in this domain — because the incoming
data had drifted from what the model was trained on.

**What you will produce**, as working, runnable code in this notebook (not
just prose):

1. `validate_schema(df, reference_stats)` — a data-contract check: correct
   columns/dtypes, no nulls, and each numeric column within an acceptable
   range learned from the training data. It should return a list of
   human-readable violation strings (empty list = valid).
2. A trained baseline pipeline (`StandardScaler` + a classifier of your
   choice) on the real dataset, with a proper train/test split.
3. `evaluate_model(model, X, y)` — returns a dict with accuracy, precision,
   recall, F1, the confusion matrix, **and specifically the recall for the
   malignant class**, since that is the metric that actually matters for
   patient safety here (overall accuracy can hide a rise in false negatives).
4. `inject_drift(df, columns, shift=2.0, scale=1.5)` — takes a copy of a
   dataframe and deliberately perturbs a few named columns (mean shift +
   variance scaling) to simulate a production data-drift incident. This is
   the one place you're allowed to use synthetic manipulation: real drift
   incidents are rare and undocumented, so to prove your detector works you
   need a case where you know the ground truth of which columns drifted.
5. `detect_drift(reference_df, new_df, alpha=0.05)` — runs a two-sample
   Kolmogorov-Smirnov test (`scipy.stats.ks_2samp`) column by column and
   returns which columns are flagged as drifted at the given significance
   level.
6. A small test suite of at least three `test_*()` functions (pytest-style:
   plain `assert` statements, no pytest dependency needed since we're in a
   notebook) that you run and report PASS/FAIL for, e.g.:
   - `test_schema_rejects_bad_data()` — a deliberately broken dataframe should
     produce violations.
   - `test_malignant_recall_above_threshold()` — the baseline model's recall
     on the malignant class should clear a minimum bar (e.g. 0.90).
   - `test_drift_detector_catches_injected_shift()` — after `inject_drift` on
     a known subset of columns, `detect_drift` should flag *those* columns
     and should **not** flag an untouched control batch (no false alarms on
     clean data).

Work through Part 1–6 below using the TODO cells. Try to get something
correct and fail-with-an-honest-error (rather than silently pass) before
peeking at the solution.

---

## Part 1 — Load the data and build a schema/reference contract

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)

RANDOM_STATE = 42

data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
# target: 0 = malignant, 1 = benign (sklearn's convention for this dataset)
TARGET_COL = "target"
MALIGNANT_LABEL = 0

df.head()

In [ ]:
# TODO: build a "reference_stats" dict describing the training data's contract.
# For each feature column (everything except TARGET_COL) you'll want at least:
#   - dtype
#   - min and max observed value (use these as the acceptable range)
# Store it as, e.g., reference_stats[col] = {"dtype": ..., "min": ..., "max": ...}
# You'll use this dict inside validate_schema() in Part 2.

reference_stats = {}
# YOUR CODE HERE
raise NotImplementedError

---

## Part 2 — Schema validation

In [ ]:
def validate_schema(frame: pd.DataFrame, reference_stats: dict) -> list[str]:
    \"\"\"Check `frame` against the training-data contract in `reference_stats`.

    Returns a list of violation strings. An empty list means the frame is valid.
    Checks to implement:
      - every expected column is present
      - no column is entirely missing / no nulls anywhere in the feature columns
      - every numeric feature column's values fall within [min, max] from
        reference_stats (a little slack is fine, but don't hardcode a huge one)
    \"\"\"
    violations = []
    # YOUR CODE HERE
    raise NotImplementedError
    return violations

---

## Part 3 — Train a baseline model

In [ ]:
# TODO: split df into train/test (stratify on the target), build a
# Pipeline(StandardScaler -> classifier), fit it on the training split.
# Keep the test split around — you'll reuse it for evaluation and for
# building "clean" vs. "drifted" batches in Part 5.

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# YOUR CODE HERE: X_train, X_test, y_train, y_test = train_test_split(...)
# YOUR CODE HERE: model = Pipeline([...]); model.fit(X_train, y_train)
raise NotImplementedError

---

## Part 4 — Evaluate with an eye for failure modes, not just accuracy

In [ ]:
def evaluate_model(model, X, y, positive_failure_label=MALIGNANT_LABEL) -> dict:
    \"\"\"Return a dict with accuracy, precision, recall, f1 (binary, treating
    `positive_failure_label` as the class we most need to catch), the full
    confusion matrix, and a `recall_malignant` entry specifically.

    Think about *why* accuracy alone would not have caught the incident
    described in the problem statement.
    \"\"\"
    # YOUR CODE HERE
    raise NotImplementedError

---

## Part 5 — Simulate drift and detect it

In [ ]:
def inject_drift(frame: pd.DataFrame, columns: list[str], shift: float = 2.0, scale: float = 1.5, random_state: int = 0) -> pd.DataFrame:
    \"\"\"Return a COPY of `frame` where each column in `columns` has been
    perturbed: new_value = (old_value * scale) + shift * std(old_value).
    Leave every other column untouched.
    \"\"\"
    # YOUR CODE HERE
    raise NotImplementedError


def detect_drift(reference_df: pd.DataFrame, new_df: pd.DataFrame, alpha: float = 0.05) -> list[str]:
    \"\"\"Run a two-sample KS test per shared numeric column between
    `reference_df` and `new_df`. Return the list of column names whose
    p-value < alpha (i.e. flagged as drifted).
    \"\"\"
    # YOUR CODE HERE
    raise NotImplementedError

---

## Part 6 — Write and run your test suite

In [ ]:
# TODO: write at least these three test functions. Each should `assert`
# and either complete silently (pass) or raise an AssertionError (fail).
# Then run them all and print a PASS/FAIL report — don't let one failure
# stop the others from running.

def test_schema_rejects_bad_data():
    raise NotImplementedError

def test_malignant_recall_above_threshold(min_recall: float = 0.90):
    raise NotImplementedError

def test_drift_detector_catches_injected_shift():
    raise NotImplementedError

tests = [test_schema_rejects_bad_data, test_malignant_recall_above_threshold, test_drift_detector_catches_injected_shift]

# YOUR CODE HERE: loop over `tests`, run each in a try/except, print PASS/FAIL per test
raise NotImplementedError

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)

RANDOM_STATE = 42
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
TARGET_COL = "target"
MALIGNANT_LABEL = 0
FEATURE_COLS = [c for c in df.columns if c != TARGET_COL]

# --- Part 1: reference contract ---
reference_stats = {
    col: {
        "dtype": str(df[col].dtype),
        "min": float(df[col].min()),
        "max": float(df[col].max()),
    }
    for col in FEATURE_COLS
}

# --- Part 2: schema validation ---
def validate_schema(frame, reference_stats):
    violations = []
    expected_cols = set(reference_stats.keys())
    actual_cols = set(frame.columns)

    missing = expected_cols - actual_cols
    if missing:
        violations.append(f"missing columns: {sorted(missing)}")

    present = expected_cols & actual_cols
    for col in present:
        if frame[col].isna().any():
            n = int(frame[col].isna().sum())
            violations.append(f"column '{col}' has {n} null value(s)")

        lo, hi = reference_stats[col]["min"], reference_stats[col]["max"]
        below = frame[col] < lo
        above = frame[col] > hi
        if below.any() or above.any():
            n_bad = int(below.sum() + above.sum())
            violations.append(
                f"column '{col}' has {n_bad} value(s) outside training range [{lo:.4f}, {hi:.4f}]"
            )
    return violations

# --- Part 3: baseline model ---
X = df[FEATURE_COLS]
y = df[TARGET_COL]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
model.fit(X_train, y_train)

# --- Part 4: evaluation beyond accuracy ---
def evaluate_model(model, X, y, positive_failure_label=MALIGNANT_LABEL):
    preds = model.predict(X)
    cm = confusion_matrix(y, preds)
    return {
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(y, preds, pos_label=positive_failure_label),
        "recall": recall_score(y, preds, pos_label=positive_failure_label),
        "f1": f1_score(y, preds, pos_label=positive_failure_label),
        "confusion_matrix": cm,
        "recall_malignant": recall_score(y, preds, pos_label=positive_failure_label),
    }

metrics = evaluate_model(model, X_test, y_test)
print(metrics)
# Why not just accuracy? With ~63% benign cases in this dataset, a model that
# mostly predicts "benign" can still score high overall accuracy while
# missing a worrying fraction of actual malignant cases. recall_malignant is
# the number that would actually catch that failure mode.

# --- Part 5: drift injection + detection ---
def inject_drift(frame, columns, shift=2.0, scale=1.5, random_state=0):
    out = frame.copy()
    for col in columns:
        std = out[col].std()
        out[col] = out[col] * scale + shift * std
    return out

def detect_drift(reference_df, new_df, alpha=0.05):
    flagged = []
    shared_cols = [c for c in reference_df.columns if c in new_df.columns]
    for col in shared_cols:
        if not np.issubdtype(reference_df[col].dtype, np.number):
            continue
        stat, p_value = ks_2samp(reference_df[col].dropna(), new_df[col].dropna())
        if p_value < alpha:
            flagged.append(col)
    return flagged

drifted_cols = ["mean radius", "mean texture", "worst area"]
clean_batch = X_test.copy()
drifted_batch = inject_drift(X_test, drifted_cols)

flags_on_clean = detect_drift(X_train, clean_batch)
flags_on_drifted = detect_drift(X_train, drifted_batch)
print("flags on clean batch:", flags_on_clean)
print("flags on drifted batch:", flags_on_drifted)

# --- Part 6: test suite ---
def test_schema_rejects_bad_data():
    broken = X_test.copy()
    broken.loc[broken.index[0], "mean radius"] = -999.0  # clearly out of range
    violations = validate_schema(broken, reference_stats)
    assert len(violations) > 0, "expected at least one violation on broken data"

def test_malignant_recall_above_threshold(min_recall=0.90):
    m = evaluate_model(model, X_test, y_test)
    assert m["recall_malignant"] >= min_recall, (
        f"malignant recall {m['recall_malignant']:.3f} fell below {min_recall}"
    )

def test_drift_detector_catches_injected_shift():
    cols = ["mean radius", "mean texture", "worst area"]
    drifted = inject_drift(X_test, cols)
    flagged = set(detect_drift(X_train, drifted))
    clean_flags = set(detect_drift(X_train, X_test.copy()))
    assert set(cols).issubset(flagged), f"expected {cols} to be flagged, got {flagged}"
    assert len(clean_flags) == 0, f"expected no false alarms on clean data, got {clean_flags}"

tests = [
    test_schema_rejects_bad_data,
    test_malignant_recall_above_threshold,
    test_drift_detector_catches_injected_shift,
]

results = []
for t in tests:
    try:
        t()
        results.append((t.__name__, "PASS", None))
    except AssertionError as e:
        results.append((t.__name__, "FAIL", str(e)))

for name, status, detail in results:
    line = f"{status}: {name}"
    if detail:
        line += f" -- {detail}"
    print(line)
```

**Why this matters for an SQA-to-ML-QE transition:** the schema check is a
standard boundary-value/contract test wearing new clothes. The drift detector
is the genuinely new skill — a *statistical* assertion, not a deterministic
one, so you need a known-good control batch (the clean one) to prove the test
doesn't cry wolf, exactly like you'd want a negative test case alongside a
positive one in classic SQA. And `recall_malignant` is the reminder that the
"metric under test" has to be chosen for the failure mode that actually
costs you, not whatever number is easiest to read off a classification
report.

</details>